# Lab 03 - Prompt Engineering und Quellenattribution (Teil III)

Citations sehen überzeugend aus, müssen aber gemessen werden. In diesem Lab
bauen Sie den Prompt vom schwachen zum belastbaren System-Prompt um, erzwingen
ein Citation-Muster, prüfen die Belege gegen den tatsächlich gelieferten
Kontext und üben die "Weiß nicht"-Strategie. Die volle Messung von
Citation-Precision und -Recall folgt in Teil IX; hier legen wir das Fundament.

## 1. Setup

In [ ]:
import re

from common import llm
from common.corpus import load_corpus
from common import retrieval as R

docs = load_corpus()
index = R.HybridRetriever(R.chunk_corpus(docs))


def hole_kontext(frage: str, k: int = 4):
    treffer = index.search(frage, k=k)
    return treffer, R.format_context(treffer)


print("Backend:", llm.active_backend())

## 2. Schwacher gegen belastbarer System-Prompt

Derselbe Kontext, dieselbe Frage, zwei System-Prompts. Der schwache lässt dem
Modell alle Freiheiten; der belastbare bindet es an den Kontext, verlangt
Belege und definiert den Ausweg "Weiß nicht".

In [ ]:
SCHWACH = "Sie sind ein hilfreicher Assistent."

BELASTBAR = (
    "Sie sind ein technischer Wissensassistent für Hydraulikpumpen. Regeln:\n"
    "1. Antworten Sie ausschließlich anhand des Kontexts.\n"
    "2. Belegen Sie jede Aussage mit der Quellenkennung in eckigen Klammern, "
    "z. B. [p12-betriebsdruck].\n"
    "3. Steht die Antwort nicht im Kontext, antworten Sie genau: "
    "'Das geht aus den Unterlagen nicht hervor.'\n"
    "4. Erfinden Sie keine Zahlen."
)

frage = "Welcher Spitzendruck ist kurzzeitig zulässig und wie lange?"
treffer, kontext = hole_kontext(frage)
prompt = f"Kontext:\n{kontext}\n\nFrage: {frage}\n\nAntwort:"

for name, sys in [("SCHWACH", SCHWACH), ("BELASTBAR", BELASTBAR)]:
    print(f"--- {name} ---")
    print(llm.complete(prompt, system=sys, temperature=0.0, max_tokens=512), "\n")

## 3. Anatomie des Prompt-Templates

Ein wiederverwendbares Template trennt die Bausteine sauber: Rolle und Regeln
(System), Belegmaterial (Kontext), Aufgabe (Frage), Formatvorgabe. Genau diese
Trennung macht den Prompt testbar und später gegen Injection härtbar
(Teil X).

In [ ]:
def baue_prompt(frage: str, kontext: str, format_hinweis: str = "") -> str:
    teile = [f"Kontext:\n{kontext}", f"Frage: {frage}"]
    if format_hinweis:
        teile.append(f"Format: {format_hinweis}")
    teile.append("Antwort:")
    return "\n\n".join(teile)


antwort = llm.complete(
    baue_prompt(frage, kontext, "Ein Satz, mit Quellenkennung am Satzende."),
    system=BELASTBAR, temperature=0.0, max_tokens=512)
print(antwort)

## 4. Citations extrahieren und gegen den Kontext prüfen

Eine Citation ist nur dann ein Beleg, wenn die zitierte Quelle wirklich im
Kontext stand. Wir ziehen die `[doc_id]`-Marken aus der Antwort und prüfen,
ob sie zu den herangezogenen Quellen gehören. Eine erfundene Kennung ist ein
Halluzinations-Signal.

In [ ]:
# Toleriert sowohl [p12-betriebsdruck] als auch ein vorangestelltes "doc_id:".
CITE_RE = re.compile(r"\[\s*(?:doc_id\s*[:=]\s*)?([a-z0-9][a-z0-9_\-]*)\s*\]")


def pruefe_citations(antwort: str, treffer) -> dict:
    zitiert = set(CITE_RE.findall(antwort))
    erlaubt = {c.doc_id for c, _ in treffer}
    return {
        "zitiert": zitiert,
        "im_kontext": zitiert & erlaubt,
        "erfunden": zitiert - erlaubt,
    }


print(pruefe_citations(antwort, treffer))

## 5. "Weiß nicht" als Prompt-Strategie

Ein ehrliches "Weiß nicht" ist eine Funktion, kein Mangel. Wir stellen eine
Frage, die der Korpus nicht beantwortet, und prüfen, ob der belastbare Prompt
die Refusal-Antwort auslöst statt zu raten.

In [ ]:
unbeantwortbar = "Welche Schmierfettsorte ist für die Antriebskupplung vorgeschrieben?"
treffer_u, kontext_u = hole_kontext(unbeantwortbar)
antwort_u = llm.complete(baue_prompt(unbeantwortbar, kontext_u),
                         system=BELASTBAR, temperature=0.0, max_tokens=512)
print(antwort_u)
print("\nRefusal erkannt:", "nicht hervor" in antwort_u.lower())

## Aufgabe 1: Prompt härten

Der belastbare Prompt verlangt Belege. Erweitern Sie ihn so, dass das Modell
bei *mehreren* Aussagen *jede einzeln* belegt und am Ende die genutzten
Quellen als Liste nennt. Testen Sie an einer Frage, deren Antwort aus zwei
Dokumenten stammt (z. B. Öl und Wechselintervall).

In [ ]:
# Ihre Lösung:

### Lösung (eine mögliche)

In [ ]:
BELASTBAR_PLUS = BELASTBAR + (
    "\n5. Belegen Sie jede Teilaussage einzeln.\n"
    "6. Schließen Sie mit einer Zeile 'Quellen: [id], [id]'."
)
frage2 = "Welches Öl ist vorgeschrieben und in welchem Intervall wird es gewechselt?"
treffer2, kontext2 = hole_kontext(frage2)
antwort2 = llm.complete(baue_prompt(frage2, kontext2), system=BELASTBAR_PLUS,
                        temperature=0.0, max_tokens=512)
print(antwort2)
print("\nCitation-Prüfung:", pruefe_citations(antwort2, treffer2))

## Aufgabe 2: Refusal robust machen

Formulieren Sie zwei unbeantwortbare Fragen, die thematisch *nah* am Korpus
liegen (gefährlicher als offensichtlich fremde Fragen). Prüfen Sie, ob der
Prompt sie zuverlässig ablehnt. Wo er einknickt: woran liegt es, am Prompt
oder am Retrieval?

In [ ]:
# Ihre Lösung:

### Lösung

In [ ]:
for f in [
    "Wie hoch ist der Schalldruckpegel der Pumpe im Betrieb?",
    "Welche Garantiezeit gilt auf den Wellendichtring als Einzelteil?",
]:
    _, ctx = hole_kontext(f)
    a = llm.complete(baue_prompt(f, ctx), system=BELASTBAR, temperature=0.0, max_tokens=512)
    print("Q:", f)
    print("A:", a)
    print("Refusal:", "nicht hervor" in a.lower(), "\n")

## Diskussion

- Was hat den größeren Effekt auf die Citation-Qualität: ein besserer
  Prompt oder ein besseres Retrieval? (Teil IX trennt beide Ursachen.)
- Eine erfundene `[doc_id]` ist leicht zu erkennen. Eine *falsch zugeordnete*
  echte Kennung nicht. Warum reicht die Existenzprüfung allein nicht?
- Welche Refusal-Formulierung würden Sie in Ihrem System als feste
  Standardantwort verankern?